In [ ]:
import numpy as np
import scipy as sp

In [ ]:
def vol_ball(radius, dim):
    """
    volume of a ball of radius `radius` in dimension `dim`
    """
    prefactor = np.pow(np.pi, dim/2)/sp.special.gamma(1 + dim/2)
    return prefactor * (radius**dim)

def frac_left(radius, dim, offset):
    """
    how much of unit sphere is cut off by a half space with offset `offset` from origin

    https://en.wikipedia.org/wiki/Spherical_cap#Hyperspherical_cap
    """
    h = radius-offset
    
    if h < 0:
        return 0

    return 0.5*sp.special.betainc(
        (dim+1)/2,
        1/2,
        (2*radius*h - h**2)/(radius**2))

# Load a CY obe

In [ ]:
from cytools import Polytope, Cone

In [ ]:
import sys; sys.path.append('..')
from src import cydata, Zp, diagnostics, lattice

In [ ]:
import sys; sys.path.append('../../cornell-dev')
from projects.kklt.kklt_lib import kklt_conifolds

In [ ]:
import numpy as np

In [ ]:
import flint

## Hard-coded Manwe's CY

In [ ]:
if True:
    # Manwe
    verts   = [[0, 0, 0, 0], [1, -1, -1, -1], [-1, 2, 1, 1], [-1, -1, 0, 0], [-1, -1, 2, 0], [-1, -1, 2, 1], [-1, 0, 0, 2], [-1, -1, 0, 2], [-1, 0, 0, 1], [-1, 0, 1, 0], [-1, -1, 0, 1], [-1, -1, 1, 0], [-1, -1, 1, 1], [-1, 0, 1, 1], [-1, 1, 1, 1], [0, -1, 0, 0]]
    heights = [0, 35, 29, 35, 31, 35, 35, 35, 15, 17, 31, 9, 21]
elif False:
    # h11=10
    verts   = [[1, 0, 0, 0], [0, 1, 0, 0], [-14, -9, -3, -1], [-3, -2, -1, 1], [0, 0, 0, 1], [0, 0, 1, 0], [-8, -5, -2, 0], [-4, -3, -1, 1], [-1, -1, 0, 1]]
    heights = [0.0, 0.0, 20.49999999999999, 0.0, -2.4999999999999964, 4.249999999999997, 6.749999999999995, 0.0, -7.999999999999995, -3.2499999999999982, -4.499999999999998, 7.249999999999999, -2.749999999999999, -5.249999999999997, 0.0]
elif True:
    # h11=16
    verts   = [[1, 0, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0], [0, 1, 0, 0], [-32, -21, -9, -1], [-10, -7, -3, 1], [-3, -2, -1, 1], [-1, -1, 0, 1]]
    heights = [0.0, 0.0, 3.500000000000006, 0.0, 60.750000000000014, 51.750000000000014, -17.749999999999996, 0.0, -18.25, 16.500000000000014, -20.25, -20.750000000000004, 10.000000000000009, 4.5000000000000036, 24.250000000000007, 0.0, -0.24999999999999506, -3.5000000000000036, 36.0, -5.000000000000006, -3.500000000000005]
else:
    # h11=20
    verts = [[1, 0, 0, 0], [0, 1, 0, 0], [-3, -2, -2, 2], [-24, -16, -6, -1], [-12, -8, -6, 3], [-9, -6, -5, 3], [-5, -3, -3, 2], [0, 0, 0, 1], [0, 0, 1, 0]]


In [ ]:
p       = Polytope(verts)
#t       = p.triangulate(heights=heights)
#cy      = t.cy()

## Set the conifold-related info

In [ ]:
# get the conifold charge
# -----------------------
conis = list(kklt_conifolds.kklt_conifolds(p.dual(), as_class=True))
assert len(conis) == 1
q = conis[0].conifold_charge()

In [ ]:
t  = conis[0].dual_triangulation()
cy = t.cy()

In [ ]:
data = cydata.CYData.from_cy(cy, coni_curve=q)

# Look at volume

In [ ]:
min_N_pts = 10000

grading = np.sum(data.H_cob, axis=0)
if data.h11<20:
    p  = Zp.mindeg_pvec_gurobi(data)
    mindeg = np.dot(p,grading)

    ps = Zp.pvecs(data, min_deg=mindeg, deg_window=max(5,mindeg//100), min_N_pts=min_N_pts)
else:
    totskc = np.rint(Cone(hyperplanes=data.H_cob).tip_of_stretched_cone()).astype(int)
    print(np.dot(totskc, grading))

    ps = Zp.pvecs(data, max_deg = 237_000)

In [ ]:
mat, Z, Binter = Zp.coniMellipsoid(ps[0], data)

In [ ]:
L = np.linalg.cholesky(mat)

In [ ]:
def vol(data, p, dilation=1):
    Q = dilation*(data.h11+data.h21+4)
    mat, Z, Binter = Zp.coniMellipsoid(ps[0], data)
    L = np.linalg.cholesky(mat)

    return vol_kernel(Q, L, Binter)

def vol_kernel(Q, L, Binter):
    # compute the volume of the slice
    # -------------------------------
    # volume of the ball |x|^2 <= Q for x = L.T c
    vol_ellipsoidal_region = vol_ball(radius=np.sqrt(Q), dim=L.shape[0])

    if True:
        # had c.T @ M @ c <= Q
        # cob using M = L @ L.T to
        # |x|^2<=Q for x = L.T c
    
        # we also had the M0 cut
        # 12 < b.T @ c
        # for b = Binter[0,:].T
        # which becomes
        # 12 < (L^-1 @ b) @ x
        u      = Binter[0,:]#np.linalg.inv(L) @ Binter[0,:]
        offset = 12

        u_norm  = np.linalg.norm(u)
        u      /= u_norm
        offset /= u_norm
    
        # so impose offset < dir.T @ x
        frac = frac_left(radius=np.sqrt(Q), dim=L.shape[0], offset=offset)
        print('frac',frac)
        vol_ellipsoidal_region *= frac
        #print(vol_ellipsoidal_region)

    # convert back to c-units
    vol_ellipsoidal_region /= np.linalg.det(L)

    # compute the volume of the lattice
    # ---------------------------------
    vol_lattice = np.linalg.det(Binter.T@Binter)

    
    # return
    return vol_ellipsoidal_region, vol_lattice

In [ ]:
a, b = vol(data, ps[0], dilation=1)

In [ ]:
num_pred = []
num_seen = []
r2s = list(range(1,200))

for r2 in r2s:
    a,b = vol_kernel(r2, np.eye(3), np.eye(3))
    num_pred.append(a/b)

    lat1, _ = lattice.fp_iterative_lincut(
        L = np.eye(3),
        Q = r2,
        linvec = np.array([0,0,1]),
        linmin = 12,#-float('inf'),
        linmax = float('inf'),
        max_N_out = 1_000_000_000,
        eps = 1e-4
    )
    lat2, _ = lattice.fp_iterative(
        L = np.eye(3),
        Q = r2,
        #linvec = np.array([0,0,1]),
        #linmin = 12,#-float('inf'),
        #linmax = float('inf'),
        max_N_out = 1_000_000_000,
        eps = 1e-4
    )
    assert len(lat1) == sum(lat2@np.array([0,0,1]) >= 12)
    num_seen.append(len(lat1))

In [ ]:
import matplotlib.pyplot as plt
plt.plot(r2s, num_pred, label='pred')
plt.plot(r2s, num_seen, label='observed')
plt.legend()

In [ ]:
num_seen[-1]

In [ ]:
vol_kernel(11, np.eye(3), np.eye(3))

In [ ]:
len(lattice.fp_ellipsoid(mat = np.eye(3), Q = 11)[0])

In [ ]:
Zp.coniZpM(
    data,
    ps[:1],
    Qmax = data.h11+data.h21+4,
    max_Kperp_gcd=1,
    M0min = -float('inf'),
    ellipsoid_dilation=1,
    cut_Kprime=False,
    verbosity=10
)